## Step 1 — Install Required Libraries


In [ ]:
!pip install -q pypdf sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 93.2 MB/s eta 0:00:00


## Step 2 — Import Libraries


In [ ]:
import pypdf
import faiss
from sentence_transformers import SentenceTransformer

print("All libraries installed successfully!")

All libraries installed successfully!


## Step 3 — Create Documents Folder


In [ ]:
import os

os.makedirs("documents", exist_ok=True)

print("Documents folder created!")

Documents folder created!


## Step 4 — Check Documents Folder


In [ ]:
os.listdir("documents")

[]

## Step 5 — Install Dataset Utility


In [ ]:
!pip install -q datasets

## Step 6 — Get Sample PDF Links


In [ ]:
import requests
import json

url = "https://huggingface.co/datasets/vectara/open_ragbench/resolve/main/pdf/arxiv/pdf_urls.json?download=true"

response = requests.get(url)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("content-type"))
print("First 100 characters:", response.text[:100])

Status code: 200
Content type: text/plain; charset=utf-8
First 100 characters: {"2407.01528v3": "https://arxiv.org/pdf/2407.01528v3", "2412.00651v1": "https://arxiv.org/pdf/2412.0


## Step 7 — Read PDF Links


In [ ]:
pdf_urls = response.json()

print("Total PDF links:", len(pdf_urls))


Total PDF links: 1000


## Step 8 — Download Sample PDF Documents


In [ ]:
import os
import requests

os.makedirs("documents", exist_ok=True)

for i, (paper_id, pdf_url) in enumerate(list(pdf_urls.items())[:5], start=1):

    file_path = f"documents/paper_{i}.pdf"

    pdf_response = requests.get(pdf_url)

    if pdf_response.status_code == 200:
        with open(file_path, "wb") as f:
            f.write(pdf_response.content)

        print(f"Downloaded: paper_{i}.pdf")
    else:
        print(f"Failed: paper_{i}.pdf")

Downloaded: paper_1.pdf
Downloaded: paper_2.pdf
Downloaded: paper_3.pdf
Downloaded: paper_4.pdf
Downloaded: paper_5.pdf


## Step 9 — Verify Downloaded PDFs


In [ ]:
pdf_files = os.listdir("documents")

print("Number of PDFs:", len(pdf_files))
print(pdf_files)

Number of PDFs: 5
['paper_2.pdf', 'paper_3.pdf', 'paper_5.pdf', 'paper_4.pdf', 'paper_1.pdf']


## Step 10 — Extract Text from PDFs


In [ ]:
from pypdf import PdfReader
import os

documents = []

for filename in os.listdir("documents"):

    if filename.endswith(".pdf"):

        file_path = os.path.join("documents", filename)
        reader = PdfReader(file_path)

        for page_number, page in enumerate(reader.pages, start=1):

            text = page.extract_text()

            if text:
                documents.append({
                    "source": filename,
                    "page": page_number,
                    "text": text
                })

print("Total pages extracted:", len(documents))

Total pages extracted: 108


## Step 11 — Inspect Extracted Text


In [ ]:
print(documents[0])

{'source': 'paper_2.pdf', 'page': 1, 'text': 'Towards Unified Molecule-Enhanced Pathology Image Representation Learning\nvia Integrating Spatial Transcriptomics\nMinghao Han1,2 Dingkang Yang1,2§ Jiabei Cheng3 Xukun Zhang1,2\nLinhao Qu4 Zizhi Chen1,2 Lihua Zhang1,2§\n1Academy for Engineering and Technology, Fudan University, Shanghai, China\n2Cognition and Intelligent Technology Laboratory, Fudan University, Shanghai, China\n3Department of Automation, Shanghai Jiaotong University, Shanghai, China\n4Fudan University, Shanghai, China\nmhhan22@m.fudan.edu.cn, dkyang20@fudan.edu.cn, lihuazhang@fudan.edu.cn\nAbstract\nRecent advancements in multimodal pre-training mod-\nels have significantly advanced computational pathology.\nHowever, current approaches predominantly rely on visual-\nlanguage models, which may impose limitations from a\nmolecular perspective and lead to performance bottlenecks.\nHere, we introduce a Unified Molecule-enhanced Pathology\nImage REpresentationn Learning framewo

## Step 12 — Install Text Splitter


In [ ]:
!pip install -q langchain-text-splitters

## Step 13 — Split Documents into Chunks


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80
)

chunks = []

for document in documents:

    text_chunks = text_splitter.split_text(document["text"])

    for chunk in text_chunks:
        chunk = str(chunk).strip()

        if chunk:
            chunks.append({
                "source": document["source"],
                "page": document["page"],
                "text": chunk
            })

print("Total chunks:", len(chunks))

Total chunks: 1097


## Step 14 — Inspect a Text Chunk


In [ ]:
print(chunks[0])

{'source': 'paper_2.pdf', 'page': 1, 'text': 'Towards Unified Molecule-Enhanced Pathology Image Representation Learning\nvia Integrating Spatial Transcriptomics\nMinghao Han1,2 Dingkang Yang1,2§ Jiabei Cheng3 Xukun Zhang1,2\nLinhao Qu4 Zizhi Chen1,2 Lihua Zhang1,2§\n1Academy for Engineering and Technology, Fudan University, Shanghai, China\n2Cognition and Intelligent Technology Laboratory, Fudan University, Shanghai, China'}


## Step 15 — Load Embedding Model


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


## Step 16 — Clean Extracted Chunks


In [ ]:
import re

clean_chunks = []

for chunk in chunks:
    text = chunk["text"]

    # Remove invalid Unicode surrogate characters
    text = text.encode("utf-8", "ignore").decode("utf-8")

    # Remove control characters
    text = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    if text:
        clean_chunks.append({
            "source": chunk["source"],
            "page": chunk["page"],
            "text": text
        })

chunks = clean_chunks

print("Clean chunks:", len(chunks))

Clean chunks: 1097


## Step 17 — Prepare Text for Embeddings


In [ ]:
texts = [chunk["text"] for chunk in chunks]

print("Total texts:", len(texts))

Total texts: 1097


## Step 18 — Validate Text


In [ ]:
for i, text in enumerate(texts):
    try:
        model.tokenizer(text)
    except Exception as e:
        print("Problem index:", i)
        print("Source:", chunks[i]["source"])
        print("Page:", chunks[i]["page"])
        print("Error:", e)
        break
else:
    print("✅ All texts are valid!")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (266 > 256). Running this sequence through the model will result in indexing errors


✅ All texts are valid!


## Step 19 — Generate Embeddings


In [ ]:
embeddings = model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True
)

print("Embeddings created successfully!")
print("Shape:", embeddings.shape)

Batches:   0%|          | 0/69 [00:00<?, ?it/s]

Embeddings created successfully!
Shape: (1097, 384)


## Step 20 — Build FAISS Vector Database


In [ ]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS index created!")
print("Total vectors:", index.ntotal)
print("Vector dimension:", dimension)

FAISS index created!
Total vectors: 1097
Vector dimension: 384


## Step 21 — Create Semantic Search


In [ ]:
def search_documents(query, top_k=3):
    if not chunks:
        return []

    top_k = min(top_k, len(chunks))

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for distance, chunk_index in zip(distances[0], indices[0]):
        if chunk_index < 0:
            continue

        results.append({
            "source": chunks[chunk_index]["source"],
            "page": chunks[chunk_index]["page"],
            "text": chunks[chunk_index]["text"],
            "distance": float(distance)
        })

    return results


## Step 22 — Install Generation Model Libraries


In [ ]:
!pip install -q transformers sentencepiece

## Step 23 — Load FLAN-T5


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer_llm = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("FLAN-T5 loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 loaded successfully!


## Step 24 — Create Answer Generation Function


In [ ]:
def generate_answer(prompt):

    inputs = tokenizer_llm(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=150
    )

    answer = tokenizer_llm.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

## Step 25 — Build Basic RAG Pipeline


In [ ]:
def rag_answer(query, top_k=3):

    results = search_documents(query, top_k)

    context = ""

    for result in results:
        context += (
            f"Source: {result['source']}, "
            f"Page: {result['page']}\n"
            f"{result['text']}\n\n"
        )

    prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Answer:
"""

    answer = generate_answer(prompt)

    return answer, results

## Step 26 — Test RAG Question Answering


In [ ]:
question = "What is the main purpose of this research?"

answer, sources = rag_answer(question)

print("ANSWER:")
print(answer)

print("\nSOURCES:")

for source in sources:
    print(f"- {source['source']} | Page {source['page']}")

ANSWER:
to ascertain the likely underlying heuristic and the primitives that drive the decisions of the DM

SOURCES:
- paper_2.pdf | Page 21
- paper_1.pdf | Page 11
- paper_1.pdf | Page 1


## Step 27 — Add Source-Aware Answers


In [ ]:
def rag_answer_with_sources(query, top_k=3):

    results = search_documents(query, top_k)

    context = ""

    for result in results:
        context += (
            f"Document: {result['source']}\n"
            f"Page: {result['page']}\n"
            f"Content: {result['text']}\n\n"
        )

    prompt = f"""
You are an intelligent document assistant.

Answer the question using ONLY the information provided in the documents.

If the answer is not available in the documents, say:
"I could not find this information in the provided documents."

Documents:
{context}

Question:
{query}

Give a clear and concise answer.
"""

    answer = generate_answer(prompt)

    return {
        "answer": answer,
        "sources": results
    }

## Step 28 — Test Source-Aware RAG


In [ ]:
result = rag_answer_with_sources(
    "What is the main purpose of this research?"
)

print("ANSWER:")
print(result["answer"])

print("\nSOURCE REFERENCES:")

for source in result["sources"]:
    print(f"📄 {source['source']} | Page {source['page']}")

ANSWER:
to ascertain the likely underlying heuristic and the primitives that drive the decisions of the DM

SOURCE REFERENCES:
📄 paper_2.pdf | Page 21
📄 paper_1.pdf | Page 11
📄 paper_1.pdf | Page 1


## Step 29 — Compare Two Documents


In [ ]:
def compare_documents(query, document1, document2, top_k=3):
    selected_results = []

    query_embedding = model.encode([query], convert_to_numpy=True).astype("float32")

    for document_name in [document1, document2]:
        document_chunks = [x for x in chunks if x["source"] == document_name]

        if not document_chunks:
            continue

        texts_for_doc = [x["text"] for x in document_chunks]
        doc_embeddings = model.encode(texts_for_doc, convert_to_numpy=True).astype("float32")

        temp_index = faiss.IndexFlatL2(doc_embeddings.shape[1])
        temp_index.add(doc_embeddings)
        distances, indices = temp_index.search(query_embedding, min(top_k, len(document_chunks)))

        for distance, idx in zip(distances[0], indices[0]):
            chunk = document_chunks[idx]
            selected_results.append({
                "source": chunk["source"],
                "page": chunk["page"],
                "text": chunk["text"],
                "distance": float(distance)
            })

    context = ""
    for result in selected_results:
        context += (
            f"Document: {result['source']}\n"
            f"Page: {result['page']}\n"
            f"Content: {result['text']}\n\n"
        )

    prompt = f"""Compare the two documents using only the provided information.

Document 1: {document1}
Document 2: {document2}

Question:
{query}

Context:
{context}

Mention important similarities and differences. If information is unavailable, say so.
"""

    answer = generate_answer(prompt)
    return answer, selected_results


## Step 30 — Test Document Comparison


In [ ]:
answer, sources = compare_documents(
    "What are the main differences between these two research papers?",
    "paper_1.pdf",
    "paper_2.pdf"
)

print("DOCUMENT COMPARISON:")
print(answer)

print("\nSOURCES:")

for source in sources:
    print(f"- {source['source']} | Page {source['page']}")

DOCUMENT COMPARISON:
BLEEP-Phikon

SOURCES:
- paper_1.pdf | Page 11
- paper_1.pdf | Page 1
- paper_1.pdf | Page 24
- paper_2.pdf | Page 21
- paper_2.pdf | Page 16
- paper_2.pdf | Page 6


## Step 31 — Add Conversational Memory


In [ ]:
chat_history = []

def chat_with_documents(query, top_k=3):

    results = search_documents(query, top_k)

    context = ""

    for result in results:
        context += (
            f"Document: {result['source']}\n"
            f"Page: {result['page']}\n"
            f"{result['text']}\n\n"
        )

    history = ""

    for chat in chat_history[-5:]:
        history += (
            f"User: {chat['question']}\n"
            f"Assistant: {chat['answer']}\n"
        )

    prompt = f"""
You are an intelligent multi-document knowledge assistant.

Use the document context and previous conversation to answer the user's question.

Previous conversation:
{history}

Document context:
{context}

Current question:
{query}

Answer clearly and concisely.
"""

    answer = generate_answer(prompt)

    chat_history.append({
        "question": query,
        "answer": answer
    })

    return answer, results

## Step 32 — Test Conversational Q&A


In [ ]:
answer, sources = chat_with_documents(
    "What is the main purpose of this research?"
)

print("Assistant:", answer)

Assistant: to ascertain the likely underlying heuristic and the primitives that drive the decisions of the DM


## Step 33 — Ask a Follow-up Question


In [ ]:
answer, sources = chat_with_documents(
    "What methodology did the researchers use?"
)

print("Assistant:", answer)

Assistant: Vir- chow: a million-slide digital pathology foundation model.


## Step 34 — Inspect Chat History


In [ ]:
print(chat_history)

[{'question': 'What is the main purpose of this research?', 'answer': 'to ascertain the likely underlying heuristic and the primitives that drive the decisions of the DM'}, {'question': 'What methodology did the researchers use?', 'answer': 'Vir- chow: a million-slide digital pathology foundation model.'}]


## Step 35 — Create Document Summarization


In [ ]:
def summarize_document(document_name, top_k=8):

    document_chunks = []

    for chunk in chunks:
        if chunk["source"] == document_name:
            document_chunks.append(chunk)

    if not document_chunks:
        return "Document not found."

    document_chunks = document_chunks[:top_k]

    context = ""

    for chunk in document_chunks:
        context += (
            f"Page {chunk['page']}:\n"
            f"{chunk['text']}\n\n"
        )

    prompt = f"""
Summarize the following document content.

Create a concise summary covering:
1. Main topic
2. Problem being addressed
3. Methodology
4. Important findings
5. Conclusion

Document content:
{context}

Summary:
"""

    summary = generate_answer(prompt)

    return summary

## Step 36 — Test Document Summarization


In [ ]:
summary = summarize_document("paper_1.pdf")

print("DOCUMENT SUMMARY")
print("=" * 60)
print(summary)

DOCUMENT SUMMARY
A random attention model with unobserved reference alternatives.


## Step 37 — Create Relevance Score


In [ ]:
def relevance_score(distance):
    # Simple display score based on FAISS L2 distance; not a probability.
    score = 1 / (1 + max(float(distance), 0.0))
    return round(score * 100, 2)


## Step 38 — RAG with Relevance Scores


In [ ]:
def rag_answer_with_score(query, top_k=3):

    results = search_documents(query, top_k)

    context = ""

    for result in results:
        context += (
            f"Document: {result['source']}\n"
            f"Page: {result['page']}\n"
            f"Content: {result['text']}\n\n"
        )

    prompt = f"""
Answer the question using ONLY the information in the documents.

Documents:
{context}

Question:
{query}

Give a clear and concise answer.
"""

    answer = generate_answer(prompt)

    for result in results:
        result["relevance_score"] = relevance_score(result["distance"])

    return answer, results

## Step 39 — Test Relevance Scores


In [ ]:
answer, results = rag_answer_with_score(
    "What is the main purpose of this research?"
)

print("ANSWER:")
print(answer)

print("\nRETRIEVED SOURCES:")

for result in results:
    print(
        f"📄 {result['source']} | "
        f"Page {result['page']} | "
        f"Relevance: {result['relevance_score']}%"
    )

ANSWER:
to ascertain the likely underlying heuristic and the primitives that drive the decisions of the DM

RETRIEVED SOURCES:
📄 paper_2.pdf | Page 21 | Relevance: 44.61%
📄 paper_1.pdf | Page 11 | Relevance: 43.56%
📄 paper_1.pdf | Page 1 | Relevance: 42.49%


## Step 40 — Save RAG Data


In [ ]:
import pickle

with open("rag_data.pkl", "wb") as f:
    pickle.dump({
        "chunks": chunks,
        "embeddings": embeddings
    }, f)

faiss.write_index(index, "faiss_index.index")

print("RAG data saved successfully!")

RAG data saved successfully!


## Step 41 — Check Saved Files


In [ ]:
import os

print(os.listdir())

['.config', 'faiss_index.index', 'documents', 'rag_data.pkl', 'sample_data']


## Step 42 — Create Flask Project Structure


In [ ]:
import os

folders = [
    "rag_project",
    "rag_project/templates",
    "rag_project/static"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully!")

for folder in folders:
    print("📁", folder)

Project folders created successfully!
📁 rag_project
📁 rag_project/templates
📁 rag_project/static


## Step 43 — Install Flask and ngrok


In [ ]:
!pip install -q flask flask-ngrok

## Step 45 — Create Web Interface


In [ ]:
%%writefile rag_project/templates/index.html

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Intelligent Multi-Document Knowledge Assistant</title>

    <style>
        * { box-sizing: border-box; }

        body {
            margin: 0;
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Arial, sans-serif;
            color: #172033;
            background:
                radial-gradient(circle at top left, rgba(79, 70, 229, 0.12), transparent 32%),
                radial-gradient(circle at bottom right, rgba(14, 165, 233, 0.12), transparent 30%),
                #f6f8fc;
            min-height: 100vh;
        }

        .page {
            width: min(1120px, 92%);
            margin: 0 auto;
            padding: 42px 0 60px;
        }

        .hero {
            text-align: center;
            margin-bottom: 28px;
        }

        .badge {
            display: inline-flex;
            align-items: center;
            gap: 7px;
            padding: 8px 14px;
            border-radius: 999px;
            background: #eef2ff;
            color: #4338ca;
            font-size: 13px;
            font-weight: 700;
            border: 1px solid #e0e7ff;
        }

        h1 {
            margin: 16px 0 10px;
            font-size: clamp(30px, 5vw, 48px);
            letter-spacing: -1.5px;
            line-height: 1.08;
        }

        .subtitle {
            max-width: 720px;
            margin: 0 auto;
            color: #667085;
            font-size: 16px;
            line-height: 1.7;
        }

        .grid {
            display: grid;
            grid-template-columns: 0.9fr 1.35fr;
            gap: 22px;
            align-items: start;
        }

        .card {
            background: rgba(255,255,255,0.92);
            border: 1px solid #e8ecf4;
            border-radius: 22px;
            padding: 24px;
            box-shadow: 0 14px 45px rgba(31, 41, 55, 0.07);
            backdrop-filter: blur(8px);
        }

        .card h2 {
            margin: 0 0 7px;
            font-size: 20px;
        }

        .card-subtitle {
            margin: 0 0 20px;
            color: #7a8498;
            font-size: 14px;
            line-height: 1.55;
        }

        .upload-box {
            border: 2px dashed #c7d2fe;
            background: #f8faff;
            border-radius: 18px;
            padding: 28px 18px;
            text-align: center;
            transition: 0.2s ease;
        }

        .upload-box:hover {
            border-color: #818cf8;
            background: #f5f7ff;
        }

        .upload-icon {
            width: 58px;
            height: 58px;
            margin: 0 auto 13px;
            display: grid;
            place-items: center;
            border-radius: 16px;
            background: #eef2ff;
            font-size: 28px;
        }

        input[type="file"] {
            width: 100%;
            margin-top: 12px;
            padding: 10px;
            border: 1px solid #dce2ec;
            border-radius: 12px;
            background: white;
        }

        .file-hint {
            margin-top: 10px;
            font-size: 12px;
            color: #8a94a6;
        }

        button {
            width: 100%;
            border: 0;
            border-radius: 13px;
            padding: 13px 18px;
            margin-top: 14px;
            font-size: 15px;
            font-weight: 700;
            cursor: pointer;
            color: white;
            background: linear-gradient(135deg, #4f46e5, #2563eb);
            box-shadow: 0 8px 20px rgba(79, 70, 229, 0.22);
            transition: transform 0.15s ease, box-shadow 0.15s ease;
        }

        button:hover {
            transform: translateY(-1px);
            box-shadow: 0 11px 25px rgba(79, 70, 229, 0.28);
        }

        textarea {
            width: 100%;
            min-height: 145px;
            padding: 15px;
            border: 1px solid #dce2ec;
            border-radius: 15px;
            resize: vertical;
            outline: none;
            font: inherit;
            line-height: 1.55;
            background: #fbfcff;
        }

        textarea:focus {
            border-color: #818cf8;
            box-shadow: 0 0 0 4px rgba(99, 102, 241, 0.10);
        }

        .tip {
            display: flex;
            gap: 9px;
            align-items: flex-start;
            margin-top: 14px;
            padding: 12px 13px;
            border-radius: 12px;
            background: #f8fafc;
            color: #667085;
            font-size: 12px;
            line-height: 1.5;
        }

        .result {
            margin-top: 22px;
            padding: 20px;
            border-radius: 17px;
            background: #f8fafc;
            border: 1px solid #e9edf5;
        }

        .result h3 {
            margin: 0 0 10px;
            font-size: 16px;
        }

        .answer-text {
            margin: 0;
            white-space: pre-wrap;
            line-height: 1.7;
            color: #30384a;
        }

        .source-list {
            list-style: none;
            padding: 0;
            margin: 0;
            display: grid;
            gap: 9px;
        }

        .source-list li {
            padding: 11px 13px;
            border-radius: 12px;
            background: white;
            border: 1px solid #e9edf5;
            color: #4b5565;
            font-size: 13px;
        }

        .source-name {
            font-weight: 700;
            color: #27324a;
        }

        .empty-state {
            text-align: center;
            color: #7a8498;
            padding: 25px 10px 5px;
            font-size: 14px;
        }

        .footer-note {
            text-align: center;
            margin-top: 22px;
            color: #98a2b3;
            font-size: 12px;
        }

        @media (max-width: 820px) {
            .grid { grid-template-columns: 1fr; }
            .page { padding-top: 25px; }
        }
    </style>
</head>

<body>
<div class="page">

    <div class="hero">
        <div class="badge">✨ AI-Powered Document Search</div>
        <h1>Intelligent Multi-Document<br>Knowledge Assistant</h1>
        <p class="subtitle">
            Upload your own PDF documents, then ask questions and get answers
            using semantic search with document and page-level sources.
        </p>
    </div>

    <div class="grid">

        <div class="card">
            <h2>📄 Add Your Documents</h2>
            <p class="card-subtitle">
                Upload one or multiple PDF files. They will be added to the
                knowledge base and can be searched immediately.
            </p>

            <form action="/upload" method="POST" enctype="multipart/form-data">
                <div class="upload-box">
                    <div class="upload-icon">📚</div>
                    <strong>Upload PDF files</strong>
                    <div class="file-hint">Select one or multiple .pdf files</div>
                    <input type="file" name="files" accept=".pdf,application/pdf" multiple required>
                </div>

                <button type="submit">⬆️ Upload & Process PDFs</button>
            </form>

            {% if upload_message %}
            <div class="tip">✅ {{ upload_message }}</div>
            {% endif %}

            {% if uploaded_files %}
            <div class="result">
                <h3>📚 Available Documents</h3>
                <ul class="source-list">
                    {% for filename in uploaded_files %}
                    <li><span class="source-name">📄 {{ filename }}</span></li>
                    {% endfor %}
                </ul>
            </div>
            {% endif %}
        </div>

        <div class="card">
            <h2>🔎 Ask Your Documents</h2>
            <p class="card-subtitle">
                Ask a question about any information contained in the uploaded
                documents.
            </p>

            <form action="/ask" method="POST">
                <textarea name="question"
                    placeholder="Example: What are the main findings of the research paper?"
                    required>{{ question or "" }}</textarea>

                <button type="submit">🤖 Ask AI</button>
            </form>

            {% if answer %}
            <div class="result">
                <h3>💡 AI Answer</h3>
                <p class="answer-text">{{ answer }}</p>
            </div>
            {% endif %}

            {% if sources %}
            <div class="result">
                <h3>📌 Sources</h3>
                <ul class="source-list">
                    {% for source in sources %}
                    <li>
                        <span class="source-name">{{ source.source }}</span>
                        — Page {{ source.page }}
                    </li>
                    {% endfor %}
                </ul>
            </div>
            {% endif %}

            {% if not answer and not sources %}
            <div class="empty-state">
                Upload PDFs on the left, then your answers will appear here.
            </div>
            {% endif %}
        </div>

    </div>

    <div class="footer-note">
        Semantic retrieval • FAISS • Sentence Transformers • FLAN-T5
    </div>
</div>
</body>
</html>


Writing rag_project/templates/index.html


## Step 46 — Create Final Flask RAG Backend


In [ ]:
%%writefile rag_project/app.py

from flask import Flask, render_template, request
import os
import pickle
import faiss
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
UPLOAD_DIR = os.path.join(BASE_DIR, "uploads")

app = Flask(__name__)
os.makedirs(UPLOAD_DIR, exist_ok=True)

# Load the existing RAG knowledge base so the original project continues to work.
with open(os.path.join(BASE_DIR, "rag_data.pkl"), "rb") as f:
    data = pickle.load(f)

chunks = list(data["chunks"])
index = faiss.read_index(os.path.join(BASE_DIR, "faiss_index.index"))

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

model_name = "google/flan-t5-base"
tokenizer_llm = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)


def rebuild_index():
    global index

    if not chunks:
        index = faiss.IndexFlatL2(384)
        return

    embeddings = embedding_model.encode(
        [chunk["text"] for chunk in chunks],
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)


def process_uploaded_pdf(file_path, filename):
    reader = PdfReader(file_path)
    new_chunks = []

    # Keep the same page-level source structure used by the original project.
    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()

        if text and text.strip():
            new_chunks.append({
                "source": filename,
                "page": page_number,
                "text": text.strip()
            })

    return new_chunks


def search_documents(query, top_k=3):
    if not chunks:
        return []

    top_k = min(top_k, len(chunks))

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for distance, chunk_index in zip(distances[0], indices[0]):
        if chunk_index < 0:
            continue

        results.append({
            "source": chunks[chunk_index]["source"],
            "page": chunks[chunk_index]["page"],
            "text": chunks[chunk_index]["text"],
            "distance": float(distance)
        })

    return results


def generate_answer(prompt):
    inputs = tokenizer_llm(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=150
    )

    return tokenizer_llm.decode(
        outputs[0],
        skip_special_tokens=True
    )


def rag_answer(query):
    results = search_documents(query)

    context = ""

    for result in results:
        context += (
            f"Document: {result['source']}\n"
            f"Page: {result['page']}\n"
            f"{result['text']}\n\n"
        )

    prompt = f"""Answer the question using ONLY the provided document context.
If the answer is not available in the context, say:
"I could not find this information in the provided documents."

Context:
{context}

Question:
{query}

Answer:
"""

    return generate_answer(prompt), results


@app.route("/")
def home():
    uploaded_files = sorted(
        [
            filename
            for filename in os.listdir(UPLOAD_DIR)
            if filename.lower().endswith(".pdf")
        ]
    )

    return render_template(
        "index.html",
        uploaded_files=uploaded_files
    )


@app.route("/upload", methods=["POST"])
def upload():
    global chunks

    files = request.files.getlist("files")

    if not files or all(not file.filename for file in files):
        return render_template(
            "index.html",
            upload_message="Please select at least one PDF file."
        )

    processed_count = 0
    skipped_files = []

    for file in files:
        if not file or not file.filename:
            continue

        filename = os.path.basename(file.filename)

        if not filename.lower().endswith(".pdf"):
            skipped_files.append(filename)
            continue

        file_path = os.path.join(UPLOAD_DIR, filename)
        file.save(file_path)

        try:
            new_chunks = process_uploaded_pdf(file_path, filename)

            if new_chunks:
                # Replace the old version of the same filename instead of
                # duplicating its pages when the user uploads it again.
                chunks = [
                    chunk for chunk in chunks
                    if chunk["source"] != filename
                ]

                chunks.extend(new_chunks)
                processed_count += 1
            else:
                skipped_files.append(filename)

        except Exception:
            skipped_files.append(filename)

    # Rebuild FAISS so newly uploaded PDFs become searchable immediately.
    rebuild_index()

    uploaded_files = sorted(
        [
            filename
            for filename in os.listdir(UPLOAD_DIR)
            if filename.lower().endswith(".pdf")
        ]
    )

    if processed_count == 0:
        message = "No PDF was processed. Please upload a readable PDF file."
    else:
        message = f"{processed_count} PDF file(s) uploaded and added to the knowledge base."

        if skipped_files:
            message += " Some files could not be processed."

    return render_template(
        "index.html",
        upload_message=message,
        uploaded_files=uploaded_files
    )


@app.route("/ask", methods=["POST"])
def ask():
    question = request.form.get("question", "").strip()

    uploaded_files = sorted(
        [
            filename
            for filename in os.listdir(UPLOAD_DIR)
            if filename.lower().endswith(".pdf")
        ]
    )

    if not question:
        return render_template(
            "index.html",
            answer="Please enter a question.",
            uploaded_files=uploaded_files
        )

    answer, sources = rag_answer(question)

    return render_template(
        "index.html",
        question=question,
        answer=answer,
        sources=sources,
        uploaded_files=uploaded_files
    )


if __name__ == "__main__":
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )


Writing rag_project/app.py


## Step 47 — Copy RAG Files to Flask Project


In [ ]:
import shutil

shutil.copy(
    "rag_data.pkl",
    "rag_project/rag_data.pkl"
)

shutil.copy(
    "faiss_index.index",
    "rag_project/faiss_index.index"
)

print("RAG files copied successfully!")

RAG files copied successfully!


### User PDF Upload Feature

The Flask interface now allows users to upload one or multiple PDF files directly from the web page. Uploaded PDFs are extracted, embedded, added to the FAISS index, and become searchable immediately. The existing RAG knowledge base remains available as well.


## Step 48 — Install pyngrok


In [ ]:
!pip install -q pyngrok

## Step 48A — Start the Flask Server


In [ ]:
import subprocess
import time

flask_process = subprocess.Popen(
    ["python", "app.py"],
    cwd="rag_project",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)
print("Flask server started on port 5000.")


Flask server started on port 5000.


## Step 49 — Create Public URL with ngrok


In [ ]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3Jll784bcRbVFtGnYwQDyQFzEsL_3VyMLHVuf7PSUbYZLXbSe"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(5000)

print("Website URL:")
print(public_url)

Website URL:
NgrokTunnel: "https://flail-uncrushed-entity.ngrok-free.dev" -> "http://localhost:5000"


## Step 50 — Complete Project Workflow
,
,**Project Name:** Intelligent Multi-Document Knowledge Assistant
,
,**Steps:**
,1. Collect PDF documents
,2. Extract text from PDFs
,3. Split text into chunks
,4. Clean the chunks
,5. Generate embeddings
,6. Store embeddings in FAISS
,7. Convert the user question into an embedding
,8. Retrieve relevant chunks
,9. Build the context prompt
,10. Generate the answer with FLAN-T5
,11. Return the answer with document/page sources
,12. Add conversational memory
,13. Add document comparison and summarization
,14. Save the RAG knowledge base
,15. Build the Flask web interface
,16. Start Flask
,17. Expose the application using ngrok
,
,**Architecture:**
,PDFs → Text Extraction → Chunking → Cleaning → Embeddings → FAISS → Retrieval → Prompt → FLAN-T5 → Answer + Sources → Flask UI → ngrok
